# GM Dataset — Functional KL Divergence Estimation

This notebook runs the **Gaussian Mixture (GM)** benchmark experiment.

It estimates KL divergence between two 1D trajectory distributions (A vs B) using Functional Flow Matching (FFM) with the MINO-T architecture.

**Pipeline:** Load trajectories → Train FFM (flow matching in function space) → Estimate KL divergence via Fourier basis projection

**Data shape:** 50000 samples × 128 timesteps × 1 dimension

In [ ]:
# imports
import os, sys
import numpy as np

# Find repo root (FKL/)
if os.path.isdir("util"):
    REPO_ROOT = os.path.abspath(".")
else:
    REPO_ROOT = os.path.abspath("..")

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from util.util import *

In [ ]:
# Configuration
GPU = "4"                        # CUDA device to use

os.chdir(REPO_ROOT)
print("Working directory:", os.getcwd())

# Paths (relative to repo root)
SCRIPTS_DIR = "scripts"
CONFIG      = "configs/gm.yaml"
DATA_DIR    = "data/GM"
LOG_DIR     = "log/GM"

M = 128  # grid size (must match config)

## Visualization

In [ ]:
data_A = np.load("data/GM/X1_data_A_128.npy")
data_B = np.load("data/GM/X1_data_B_128.npy")

print(f"A shape: {data_A.shape}")
print(f"B shape: {data_B.shape}")

In [ ]:
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
})

N_LINES = 200
ALPHA = 0.15
LW = 0.6

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

tau = np.linspace(0, 1, M)

rng = np.random.default_rng(42)
n_show = min(N_LINES, data_A.shape[0], data_B.shape[0])
idx_A = rng.choice(data_A.shape[0], size=n_show, replace=False)
idx_B = rng.choice(data_B.shape[0], size=n_show, replace=False)

for i in idx_A:
    axes[0].plot(tau, data_A[i, :, 0] if data_A.ndim == 3 else data_A[i, :],
                color="tab:blue", alpha=ALPHA, lw=LW)
axes[0].set_title("Distribution A")
axes[0].set_xlabel("$\\tau$")
axes[0].set_ylabel("Value")

for i in idx_B:
    axes[1].plot(tau, data_B[i, :, 0] if data_B.ndim == 3 else data_B[i, :],
                color="tab:orange", alpha=ALPHA, lw=LW)
axes[1].set_title("Distribution B")
axes[1].set_xlabel("$\\tau$")

plt.tight_layout()
plt.show()

## Run A vs B

In [ ]:
!cd {SCRIPTS_DIR} && \
  CUDA_VISIBLE_DEVICES={GPU} python main.py \
    -c ../{CONFIG} \
    --set data_params.X1.A_path.{M}=../{DATA_DIR}/X1_data_A_{M}.npy \
    --set data_params.X1.B_path.{M}=../{DATA_DIR}/X1_data_B_{M}.npy \
    --set trainer_params.sfolder={LOG_DIR}/A_vs_B

## Inspect results

Load the output config files which contain the estimated KL divergence values.

In [ ]:
import yaml
from pathlib import Path

run_dir = Path(LOG_DIR) / "A_vs_B"
result_files = sorted(run_dir.glob("config_kl_FINAL_*.yaml"))

if not result_files:
    print("No KL result files found yet.")
else:
    with open(result_files[-1], "r") as f:
        cfg = yaml.safe_load(f)

    kl = cfg.get("kl_result", {}).get(1, {})
    fwd = kl.get("forward_KL", "N/A")
    rev = kl.get("reverse_KL", "N/A")

    fwd_str = f"{fwd:.4f}" if isinstance(fwd, (int, float)) else str(fwd)
    rev_str = f"{rev:.4f}" if isinstance(rev, (int, float)) else str(rev)

    print(f"{'Comparison':<16} {'Forward KL':>12} {'Reverse KL':>12}")
    print("-" * 44)
    print(f"{'A vs B':<16} {fwd_str:>12} {rev_str:>12}")

## View generated trajectory plots

Display the generated vs real trajectory PDFs.

In [ ]:
from IPython.display import display, Image
from pathlib import Path
import subprocess

run_dir = Path(LOG_DIR) / "A_vs_B" / "imgs"

for pdf in sorted(run_dir.glob("*.pdf")):
    if "energyspectrum" in pdf.name:
        continue
    png = pdf.with_suffix(".png")
    subprocess.run(["pdftoppm", "-png", "-r", "150", "-singlefile", str(pdf), str(png.with_suffix(""))], check=True)
    print(f"\n{pdf.name}")
    display(Image(filename=str(png)))